In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
import pickle


df = pd.read_csv('C:/Users/Laiba/Downloads/merged_sentiment_data_2nd_iteration_10K (1).csv')

# Preprocess
df = df[['tweet_text', 'context', 'sentiment']]

# Handle NaN values
df['tweet_text'] = df['tweet_text'].fillna('').astype(str).str.lower()
df['context'] = df['context'].fillna('').astype(str).str.lower()
df = df.dropna(subset=['sentiment'])

# Combine tweet text and context for training
df['combined_text'] = df['tweet_text'] + ' [SEP] ' + df['context']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(df[['combined_text', 'tweet_text']], df['sentiment'], test_size=0.2, random_state=42)

# Create a pipeline for the combined text (used only for training)
train_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000)),
    ('classifier', SVC(kernel='rbf', C=1.0, random_state=42))
])

# Train the model on combined text
train_pipeline.fit(X_train['combined_text'], y_train)

# Create a separate pipeline for tweet text only (used for prediction)
predict_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000)),
    ('classifier', SVC(kernel='rbf', C=1.0, random_state=42))
])

# Fit the TfidfVectorizer on tweet_text only
predict_pipeline.named_steps['tfidf'].fit(X_train['tweet_text'])

# Transform the tweet_text using the fitted TfidfVectorizer
X_train_tweet_tfidf = predict_pipeline.named_steps['tfidf'].transform(X_train['tweet_text'])

# Train the classifier on the transformed tweet_text
predict_pipeline.named_steps['classifier'].fit(X_train_tweet_tfidf, y_train)

# Evaluate the model
y_pred = predict_pipeline.predict(X_test['tweet_text'])
print(classification_report(y_test, y_pred))



              precision    recall  f1-score   support

    negative       0.74      0.79      0.76       719
     neutral       0.66      0.59      0.62       697
    positive       0.70      0.72      0.71       613

    accuracy                           0.70      2029
   macro avg       0.70      0.70      0.70      2029
weighted avg       0.70      0.70      0.70      2029



In [15]:

def predict_sentiment(tweet):
    return predict_pipeline.predict([tweet])[0]

# Example usage
new_tweet = "This product is okay, but could be better."
print(f"Sentiment: {predict_sentiment(new_tweet)}")

with open('sentiment_model.pkl', 'wb') as file:
    pickle.dump(predict_pipeline, file)

print("Model saved to sentiment_model.pkl")


new_tweet = "This product is amazing!"
print(f"Sentiment: {loaded_model.predict([new_tweet])[0]}")

# Example usage
new_tweet = "it's 80 degrees in october but donald trump says climate change is a hoax made up by the chinese…"
print(f"Sentiment: {predict_sentiment(new_tweet)}")
 
# Example usage
new_tweet = "Look at the stupidity of these scumbags, as if they were going to invade India or teach Israel a lesson."
print(f"Sentiment: {predict_sentiment(new_tweet)}")
 
# Example usage
new_tweet = "You need to get your hands on this ice cream that fights climate change! #saveourswirled ."
print(f"Sentiment: {predict_sentiment(new_tweet)}")
 
# Example usage
new_tweet = "OMG! This New AI Feature in One UI 6.1.1 is INSANE https://t.co/bLP0ejPCZP via @YouTube #OneUI6.1.1 #OneUI6 #SamsungUpdate #NewFeatures #TechUpdates #Samsung #Galaxy #Smartphone #Android #Mobile #Gadget #Tech #Update #Software #UI #Design #Tips #satech #satechsunny #sunnysatech"
print(f"Sentiment: {predict_sentiment(new_tweet)}")
 
# Example usage
new_tweet = "i am going through a lot these days , but still i am surviving."
print(f"Sentiment: {predict_sentiment(new_tweet)}")
 
# Example usage
new_tweet = ("I live in CA. It's a great place to live. Beautiful coastline, all affluent cities that vote Rep. "
             "YES we have had bad governors like Arnold and Reagan who spent the budget Brown balanced. "
             "No humidity, no crime, no homeless. YES, LA and ALL big cities suck .")

print(f"Sentiment: {predict_sentiment(new_tweet)}")

Sentiment: neutral
Model saved to sentiment_model.pkl
Sentiment: positive
Sentiment: negative
Sentiment: negative
Sentiment: positive
Sentiment: positive
Sentiment: positive
Sentiment: negative


In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
import pickle


# Save the predict_pipeline to a file
with open('sentiment_model.pkl', 'wb') as file:
    pickle.dump(predict_pipeline, file)

print("Model saved to sentiment_model.pkl")

# Test tweets
test_tweets = {
    'negative': [
        "This product is absolutely terrible. I regret buying it.",
        "The customer service was awful. I'll never use this company again.",
        "I'm extremely disappointed with the quality of this item.",
        "The movie was a complete waste of time and money.",
        "This restaurant has the worst food I've ever tasted.",
        "I can't believe how poorly designed this app is.",
        "The hotel room was dirty and uncomfortable. Worst stay ever.",
        "This book is so boring, I couldn't even finish it.",
        "The concert was a disaster. The sound quality was terrible.",
        "I'm furious with how this situation was handled."
    ],
    'neutral': [
        "The product works as expected. Nothing special.",
        "I neither liked nor disliked the movie. It was okay.",
        "The service was average. Not great, not terrible.",
        "The food was edible, but I wouldn't go out of my way to eat there again.",
        "This book had its ups and downs. Overall, it was mediocre.",
        "The hotel room was standard. Clean but basic.",
        "The app functions, but it could use some improvements.",
        "The concert was alright. Some songs were good, others not so much.",
        "I have mixed feelings about this purchase.",
        "It's hard to form a strong opinion about this product."
    ],
    'positive': [
        "I absolutely love this product! It exceeded my expectations.",
        "The customer service was outstanding. They went above and beyond.",
        "This movie was fantastic! I highly recommend it.",
        "The food at this restaurant is delicious. I can't wait to go back.",
        "This book was a page-turner. I couldn't put it down!",
        "The hotel stay was perfect. The staff was incredibly friendly.",
        "This app is so user-friendly and helpful. Great job!",
        "The concert was amazing! The best live performance I've seen.",
        "I'm thoroughly impressed with the quality of this item.",
        "This experience made my day. I'm so happy!"
    ]
}

# Function to test the model on a list of tweets
def test_model(tweets, expected_sentiment):
    results = []
    for tweet in tweets:
        sentiment = predict_sentiment(tweet)
        results.append({
            'tweet': tweet,
            'predicted': sentiment,
            'expected': expected_sentiment,
            'correct': sentiment == expected_sentiment
        })
    return results

# Test the model on each category
for sentiment, tweets in test_tweets.items():
    print(f"\nTesting {sentiment} tweets:")
    results = test_model(tweets, sentiment)
    correct = sum(1 for r in results if r['correct'])
    print(f"Accuracy: {correct}/{len(results)} ({correct/len(results)*100:.2f}%)")
    
    # Print details of incorrect predictions
    for r in results:
        if not r['correct']:
            print(f"\nIncorrect prediction:")
            print(f"Tweet: {r['tweet']}")
            print(f"Predicted: {r['predicted']}")
            print(f"Expected: {r['expected']}")

# Overall accuracy
all_results = [r for results in [test_model(tweets, sent) for sent, tweets in test_tweets.items()] for r in results]
total_correct = sum(1 for r in all_results if r['correct'])
print(f"\nOverall Accuracy: {total_correct}/{len(all_results)} ({total_correct/len(all_results)*100:.2f}%)")

Model saved to sentiment_model.pkl

Testing negative tweets:
Accuracy: 5/10 (50.00%)

Incorrect prediction:
Tweet: This product is absolutely terrible. I regret buying it.
Predicted: positive
Expected: negative

Incorrect prediction:
Tweet: The customer service was awful. I'll never use this company again.
Predicted: positive
Expected: negative

Incorrect prediction:
Tweet: I'm extremely disappointed with the quality of this item.
Predicted: positive
Expected: negative

Incorrect prediction:
Tweet: This restaurant has the worst food I've ever tasted.
Predicted: positive
Expected: negative

Incorrect prediction:
Tweet: I can't believe how poorly designed this app is.
Predicted: positive
Expected: negative

Testing neutral tweets:
Accuracy: 1/10 (10.00%)

Incorrect prediction:
Tweet: The product works as expected. Nothing special.
Predicted: positive
Expected: neutral

Incorrect prediction:
Tweet: I neither liked nor disliked the movie. It was okay.
Predicted: positive
Expected: neutral
